In [ ]:
# !pip install google-api-python-client

In [1]:
import importlib.metadata

# List the distribution package names
packages = ["google-api-python-client"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


google-api-python-client version: 2.198.0


In [1]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

True

## Read public calendar - Need API key

In [2]:
import os
from googleapiclient.discovery import build

# Configuration
API_KEY = os.getenv("GOOGLE_CALENDAR_API_KEY")

# Build service
service = build(
    "calendar",
    "v3",
    developerKey=API_KEY
)

# Public US Holidays calendar
# CALENDAR_ID = "en.usa#holiday@group.v.calendar.google.com"
CALENDAR_ID = "en.indian#holiday@group.v.calendar.google.com"


# Get next 5 events
events_result = service.events().list(
    calendarId=CALENDAR_ID,
    maxResults=5,
    singleEvents=True,
    orderBy="startTime"
).execute()

events = events_result.get("items", [])

if not events:
    print("No upcoming events found.")

for event in events:
    start = event["start"].get("dateTime", event["start"].get("date"))
    print(f"{start} - {event['summary']}")

2021-01-01 - New Year's Day
2021-01-13 - Lohri
2021-01-14 - Makar Sankranti
2021-01-14 - Pongal
2021-01-20 - Guru Govind Singh Jayanti


## I want this year holidays

In [3]:
import os
from datetime import datetime
from googleapiclient.discovery import build

# Configuration
API_KEY = os.getenv("GOOGLE_CALENDAR_API_KEY")

# CALENDAR_ID = "en.usa#holiday@group.v.calendar.google.com"
CALENDAR_ID = "en.indian#holiday@group.v.calendar.google.com"

# Build service
service = build("calendar", "v3", developerKey=API_KEY)

# Define the start and end of the current year (2026) in RFC3339 format
current_year = datetime.now().year  # Dynamically gets 2026
time_min = f"{current_year}-01-01T00:00:00Z"
time_max = f"{current_year}-12-31T23:59:59Z"

# Get all events for this year
events_result = (
    service.events()
    .list(
        calendarId=CALENDAR_ID,
        timeMin=time_min,
        timeMax=time_max,
        singleEvents=True,
        orderBy="startTime",
    )
    .execute()
)

events = events_result.get("items", [])

if not events:
    print(f"No upcoming events found for {current_year}.")
else:
    print(f"--- Public Holidays for {current_year} ---")
    for event in events:
        start = event["start"].get("dateTime", event["start"].get("date"))
        print(f"{start} - {event['summary']}")


--- Public Holidays for 2026 ---
2026-01-01 - New Year's Day
2026-01-03 - Hazarat Ali's Birthday
2026-01-14 - Pongal
2026-01-14 - Makar Sankranti
2026-01-23 - Vasant Panchami
2026-01-26 - Republic Day
2026-02-01 - Guru Ravidas Jayanti
2026-02-12 - Maharishi Dayanand Saraswati Jayanti
2026-02-15 - Maha Shivaratri
2026-02-19 - Ramadan Start
2026-02-19 - Shivaji Jayanti
2026-03-03 - Holika Dahana
2026-03-04 - Holi
2026-03-19 - Ugadi
2026-03-19 - Gudi Padwa
2026-03-20 - Jamat Ul-Vida
2026-03-21 - Ramzan Id
2026-03-26 - Rama Navami
2026-03-31 - Mahavir Jayanti
2026-04-03 - Good Friday
2026-04-05 - Easter Day
2026-04-14 - Vaisakhi
2026-04-14 - Ambedkar Jayanti
2026-04-14 - Mesadi
2026-04-15 - Bahag Bihu
2026-05-01 - Buddha Purnima
2026-05-09 - Birthday of Rabindranath
2026-05-28 - Bakrid
2026-06-26 - Muharram/Ashura (tentative)
2026-07-16 - Rath Yatra
2026-08-15 - Independence Day
2026-08-26 - Onam
2026-08-26 - Milad un-Nabi (tentative)
2026-08-28 - Raksha Bandhan
2026-09-04 - Janmashtami
20

## Read your calendar event: You need credentials.json in CWD

In [9]:
from datetime import datetime, timezone
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os

SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]

creds = None

if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json",
            SCOPES
        )
        creds = flow.run_local_server(port=0)

    with open("token.json", "w") as token:
        token.write(creds.to_json())

service = build("calendar", "v3", credentials=creds)

now = datetime.now(timezone.utc).isoformat()

events = (
    service.events()
    .list(
        calendarId="primary",
        timeMin=now,
        maxResults=10,
        singleEvents=True,
        orderBy="startTime",
    )
    .execute()
)

for event in events.get("items", []):
    start = event["start"].get("dateTime", event["start"].get("date"))
    print(start, "-", event["summary"])

    start = datetime.fromisoformat(event["start"]["dateTime"])
    end = datetime.fromisoformat(event["end"]["dateTime"])
    
    duration = end - start
    hours = duration.seconds // 3600
    minutes = (duration.seconds % 3600) // 60
    
    print(f"""
    Title    : {event['summary']}
    Starts   : {start:%d %b %Y %I:%M %p}
    Ends     : {end:%d %b %Y %I:%M %p}
    Duration : {hours} hour(s) {minutes} minute(s)
    """)

    

2026-07-11T18:00:00+05:30 - GenAI Class

    Title    : GenAI Class
    Starts   : 11 Jul 2026 06:00 PM
    Ends     : 11 Jul 2026 09:00 PM
    Duration : 3 hour(s) 0 minute(s)
    
